In [ ]:
# Install dependencies for this notebook

try:
    import uproot
    import awkward as ak
    import hist
    import vector
    import awkward_pandas
except ModuleNotFoundError:
    import sys

    if sys.platform.startswith("emscripten"):
        import micropip

        await micropip.install("awkward==2.8.5")
        await micropip.install("uproot")
        await micropip.install("hist")
        await micropip.install("awkward-pandas")
        await micropip.install("vector")
    else:
        %pip install uproot awkward hist awkward_pandas vector

# Now let's take a quick look at a few other packages in Scikit-HEP

## Lorentz vectors

In keeping with the “many small packages” philosophy, 2D/3D/Lorentz vectors are handled by a package named Vector. This is where you can find calculations like `deltaR` and coordinate transformations.

In [ ]:
import vector

one = vector.obj(px=1, py=0, pz=0)
two = vector.obj(px=0, py=1, pz=1)

In [ ]:
one + two

In [ ]:
one.deltaR(two)

In [ ]:
one.to_rhophieta()

In [ ]:
two.to_rhophieta()

To fit in with the rest of the ecosystem, Vector must be an array-oriented library. Arrays of 2D/3D/Lorentz vectors are processed in bulk.

MomentumNumpy2D, MomentumNumpy3D, MomentumNumpy4D are NumPy array subtypes: NumPy arrays can be cast to these types and get all the vector functions.

In [ ]:
import skhep_testdata
import uproot
import awkward as ak
import vector

tree = uproot.open("data/uproot-Zmumu.root")["events"]

one = ak.to_numpy(tree.arrays(filter_name=["E1", "p[xyz]1"]))
two = ak.to_numpy(tree.arrays(filter_name=["E2", "p[xyz]2"]))

In [ ]:
one.dtype.names = ("E", "px", "py", "pz")
two.dtype.names = ("E", "px", "py", "pz")

In [ ]:
one = one.view(vector.MomentumNumpy4D)
two = two.view(vector.MomentumNumpy4D)

In [ ]:
one + two

In [ ]:
one.deltaR(two)

In [ ]:
one.to_rhophieta()

After `vector.register_awkward()` is called, `"Momentum2D"`, `"Momentum3D"`, `"Momentum4D"` are record names that Awkward Array will recognize to get all the vector functions.

In [ ]:
vector.register_awkward()

tree = uproot.open(skhep_testdata.data_path("uproot-HZZ.root"))["events"]

array = tree.arrays(filter_name=["Muon_E", "Muon_P[xyz]"])

muons = ak.zip(
    {"px": array.Muon_Px, "py": array.Muon_Py, "pz": array.Muon_Pz, "E": array.Muon_E},
    with_name="Momentum4D",
)
mu1, mu2 = ak.unzip(ak.combinations(muons, 2))

In [ ]:
mu1 + mu2

In [ ]:
mu1.deltaR(mu2)

## Particle properties and PDG identifiers

The Particle library provides all of the particle masses, decay widths and more from the PDG. It further contains a series of tools to programmatically query particle properties and use several identification schemes.

In [ ]:
import particle
from hepunits import GeV

In [ ]:
particle.Particle.findall("pi")

In [ ]:
z_boson = particle.Particle.from_name("Z0")
z_boson.mass / GeV, z_boson.width / GeV

In [ ]:
print(z_boson.describe())

In [ ]:
particle.Particle.from_pdgid(111)

In [ ]:
particle.Particle.findall(
    lambda p: p.pdgid.is_meson and p.pdgid.has_strange and p.pdgid.has_charm
)

In [ ]:
print(particle.PDGID(211).info())

## Jet clustering

In a high-energy pp collision, for instance, a spray of hadrons is produced which is clustered into "jets" of particles and this method/process is called jet-clustering. The anti-kt jet clustering algorithm is one such algorithm used to combine particles/hadrons that are close to each other into jets.

Some people need to do jet-clustering at the analysis level. The fastjet package makes it possible to do that an (Awkward) array at a time.

In [ ]:
import skhep_testdata
import uproot
import awkward as ak
import particle
from hepunits import GeV
import vector

vector.register_awkward()

picodst = uproot.open(
    "https://pivarski-princeton.s3.amazonaws.com/pythia_ppZee_run17emb.picoDst.root:PicoDst"
)
px, py, pz = ak.unzip(
    picodst.arrays(filter_name=["Track/Track.mPMomentum[XYZ]"], entry_stop=100)
)

probable_mass = particle.Particle.from_name("pi+").mass / GeV

pseudojets = ak.zip(
    {"px": px, "py": py, "pz": pz, "mass": probable_mass}, with_name="Momentum4D"
)
good_pseudojets = pseudojets[pseudojets.pt > 0.1]

import sys

if sys.platform.startswith("emscripten"):
    print(
        "`fastjet` is not available in Pyodide. You need a local Python installation for this. Skipping..."
    )
else:
    import fastjet

    jetdef = fastjet.JetDefinition(fastjet.antikt_algorithm, 1.0)

    clusterseq = fastjet.ClusterSequence(good_pseudojets, jetdef)
    clusterseq.inclusive_jets()

    ak.num(good_pseudojets), ak.num(clusterseq.inclusive_jets())

This fastjet package uses Vector to get coordinate transformations and all the Lorentz vector methods you’re accustomed to having in pseudo-jet objects. I used Particle to impute the mass of particles with only track-level information.

See how all the pieces accumulate?

# Scaling up

The tools described today are intended to be used within a script that is scaled up for large datasets.

The Coffea project collects provides an easy way to use all these tools in a distributed way so that you can perform large data analysis on a large compute grid.

:::{figure} ./images/coffea_logo.svg
:align: center
:width: 40%
:::

This topic is too large to cover here, but we refer you to the [Coffea documentation](https://coffea-hep.readthedocs.io).

# Summary

- There are a variety of Python tools for HEP, each with a very specific task.
- All these tools work together to create an ecosystem.
- You have the flexibility of picking whatever is right for your analysis.

You can explore the full list of packages at <https://scikit-hep.org>. You can contribute to making them even better by let us know about any issues you find or features you want to see, or by submitting a PR! 